#### Exercise 14 - Starter - Validate Regression Discontinuity

In [3]:
# Import pandas as statsmodels.formula.api 
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy import stats


In [2]:
# TO-DO: Import rd_employee_sales_data.csv and print the share of part-time employees by training status 
# Are the two groups comparable?  

df = pd.read_csv('../data/rd_employee_sales_data.csv', parse_dates=True)
df.head()

# TO-DO: Using scipy.stats, perform a t-test to compare the average share of employees that are part-time
# among those who have received training versus those who have not.

,EmployeeID,Skill,Year1_Sales,Training,Year2_Sales,Part_Time
0,1,69.934283,41.963919,1,57.832781,0
1,2,57.234714,33.240525,0,19.688025,0
2,3,72.953771,36.775037,0,34.678737,0
3,4,90.460597,41.995615,1,66.735835,0
4,5,55.316933,31.149583,0,28.577639,0


In [14]:
df.groupby(['Training'])['Part_Time'].value_counts(normalize=True)

Training  Part_Time
0         0            0.71798
          1            0.28202
1         0            1.00000
Name: proportion, dtype: float64

In [12]:
t, p= stats.ttest_ind(df[df['Training']==0]['Part_Time'], df[df['Training']==1]['Part_Time'], equal_var=False)
print(f'P value: {p:.3f}')

P value: 0.000


In [13]:
cutoff = 40

# TO-DO: Subset only keeping employees within a +/- 10 window of Year1_Sales around the cutoff 
# and print the share of part-time employees by training status for this subset
df['RunningVar']=df['Year1_Sales']-cutoff

subset=df[(df['RunningVar']>=-10)&(df['RunningVar']<=10)]

subset.groupby(['Training'])['Part_Time'].value_counts(normalize=True)

Training  Part_Time
0         0            1.0
1         0            1.0
Name: proportion, dtype: float64

In [19]:
# TO-DO: Run a placebo RD model completely below the true cutoff

# TO-DO: Subset the data to only include employees with Year1_Sales below the cutoff: 
# This excludes any employees who received training

subset=df[df['RunningVar']<cutoff]

# TO-DO: Define a new cutoff for the placebo test and create a binary variable for training status (if Year1_Sales >= placebo_cutoff)
placebo_cutoff = 30 

subset['placebo_training']=(subset['Year1_Sales']>placebo_cutoff).astype(int)

# TO-DO: Subset the placebo data to only include employees within a +/- 10 window of Year1_Sales around the placebo cutoff 
subset['placebo_running']=subset['Year1_Sales']-placebo_cutoff
placebo_subset = subset[(subset['placebo_running']>=-10) & (subset['placebo_running']<=10)]

# TO-DO: Run the placebo RD model using the placebo subset and the placebo cutoff 
rd_model = smf.ols(formula='Year2_Sales~placebo_running+placebo_training', data=placebo_subset).fit()

print(rd_model.summary())

                            OLS Regression Results                            
Dep. Variable:            Year2_Sales   R-squared:                       0.365
Model:                            OLS   Adj. R-squared:                  0.363
Method:                 Least Squares   F-statistic:                     189.7
Date:                Mon, 15 Dec 2025   Prob (F-statistic):           8.02e-66
Time:                        14:37:56   Log-Likelihood:                -2196.9
No. Observations:                 664   AIC:                             4400.
Df Residuals:                     661   BIC:                             4413.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           29.6802      0.569  

## Validating the Local Randomization Assumption
Across all employees, we started by asking whether treated employees had the same characteristics as untreated employees. We specifically looked at the Part_Time attribute, a covariate that indicates whether the employee was working part-time or full-time.

We determined that 28% of the untreated employees were part-time, while 0% of the treated employees were part-time. A t-test showed that this is a statistically significant difference. This indicated a violation of the local randomization assumption—if we used the full dataset in our model.

However, when we selected only the subset of employees within a bandwidth of plus or minus 10 around the cutoff, we saw that 0% of employees were part-time in both the treated and untreated groups. This lack of difference supported our local randomization assumption.

Note that with a real-world dataset, you will probably have more than one attribute to consider, but we only looked at one to simplify the problem.

## Validating the Continuity Assumption
We removed all treated employees and set a placebo cutoff of 30, with a bandwidth of plus or minus 10. With the placebo model, the p-value for Training was 0.700, meaning that the impact of the placebo "training" was not statistically different from zero. This finding supported our continuity assumption because it failed to find a discontinuity at the placebo cutoff.